# 🏁 Notebook 3 — Race Winner Prediction Model
**Formula 1 ML Analytics Project**

**Question answered**: *Who is likely to win a specific race?*

Models built:
- **Model 1a**: Binary classifier — `is_win` (XGBoost, LightGBM, Random Forest)
- **Model 1b**: Regression — `positionOrder`
- **Tuning**: Optuna with 100 trials, TimeSeriesSplit CV
- **Split**: Train 2000–2021, Val 2022, Test 2023–2024

**Key finding from prior statistical analysis:**
- Grid position → race finish: Pearson r = 0.828 (p < 0.001)
- Constructor identity explains 31% of finish variation (ANOVA ω² = 0.31)


In [ ]:
import os, sys, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

sys.path.insert(0, "..")
DATA_PATH = "../data/processed/"
MODEL_PATH = "../models/"
os.makedirs(MODEL_PATH, exist_ok=True)

# Load featured data
feat_file = os.path.join(DATA_PATH, "featured_df.csv")
if not os.path.exists(feat_file):
    print("⚠️  featured_df.csv not found. Run notebooks 01 and 02 first.")
else:
    featured_df = pd.read_csv(feat_file, low_memory=False)
    print(f"✅ Loaded featured_df: {featured_df.shape}")


In [ ]:
from src.models import (
    get_time_split, prepare_features, train_race_winner_model,
    evaluate_model, save_model, load_model, FEATURE_COLS
)

# Time-based train/val/test split
train_df, val_df, test_df = get_time_split(featured_df, train_end_year=2021, val_year=2022)
print(f"Train:  {len(train_df):,} rows | years {train_df['year'].min()}–{train_df['year'].max()}")
print(f"Val:    {len(val_df):,} rows  | year {val_df['year'].min()}–{val_df['year'].max()}")
print(f"Test:   {len(test_df):,} rows  | years {test_df['year'].min()}–{test_df['year'].max()}")
print(f"\nWin rate in train: {train_df['is_win'].mean():.1%}" if 'is_win' in train_df.columns else "")


In [ ]:
# Prepare feature matrices
X_train, y_train_win, y_train_pos = prepare_features(train_df)
X_val,   y_val_win,   y_val_pos   = prepare_features(val_df)
X_test,  y_test_win,  y_test_pos  = prepare_features(test_df)

print(f"X_train shape: {X_train.shape}")
print(f"Win rate (train): {y_train_win.mean():.1%}")
print(f"Win rate (test):  {y_test_win.mean():.1%}")


In [ ]:
# Train XGBoost model with Optuna tuning
# Set use_optuna=False and n_trials=20 for faster testing
print("Training XGBoost race winner model...")
print("(Optuna tuning: ~5-10 minutes with 100 trials; set use_optuna=False for quick run)")

xgb_model = train_race_winner_model(
    X_train, y_train_win,
    model_type='xgboost',
    use_optuna=True,
    n_trials=50   # Reduce to 20 for speed; increase to 100 for best results
)
print("✅ XGBoost model trained.")


In [ ]:
# Evaluate on validation set
print("=== VALIDATION SET PERFORMANCE ===")
val_metrics = evaluate_model(xgb_model, X_val, y_val_win, y_val_pos)
for k, v in val_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


In [ ]:
# Evaluate on test set (2023–2024)
print("=== TEST SET PERFORMANCE (2023-2024) ===")
test_metrics = evaluate_model(xgb_model, X_test, y_test_win, y_test_pos)
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


In [ ]:
# Compare all three model types
print("Training LightGBM and Random Forest for comparison...")
lgb_model = train_race_winner_model(X_train, y_train_win, model_type='lightgbm', use_optuna=False)
rf_model  = train_race_winner_model(X_train, y_train_win, model_type='random_forest', use_optuna=False)

models = {'XGBoost': xgb_model, 'LightGBM': lgb_model, 'RandomForest': rf_model}
print("\n=== MODEL COMPARISON (TEST SET) ===")
print(f"{'Model':<15} {'ROC-AUC':>10} {'Prec@1':>10} {'Top3-Acc':>10} {'LogLoss':>10}")
print("-" * 55)
for name, m in models.items():
    mm = evaluate_model(m, X_test, y_test_win, y_test_pos)
    print(f"{name:<15} {mm.get('roc_auc',0):>10.4f} {mm.get('precision_at_1',0):>10.4f} {mm.get('top3_accuracy',0):>10.4f} {mm.get('log_loss',0):>10.4f}")


In [ ]:
# SHAP feature importance
try:
    import shap
    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_val)
    
    plt.figure(figsize=(10, 8))
    feature_names = [c for c in FEATURE_COLS if c in featured_df.columns]
    shap.summary_plot(shap_values, X_val, feature_names=feature_names, 
                      max_display=20, show=False)
    plt.title("SHAP Feature Importance — Race Winner Model", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_PATH, "shap_summary.png"), dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ SHAP summary plot saved.")
except Exception as e:
    print(f"SHAP plot skipped: {e}")


In [ ]:
# Save the best model
save_model(xgb_model, os.path.join(MODEL_PATH, "race_winner_model.pkl"))
print("✅ Model saved to", os.path.join(MODEL_PATH, "race_winner_model.pkl"))


In [ ]:
# -----------------------------------------------------------------------
# USER-FACING PREDICTION: Who wins at a specific race?
# -----------------------------------------------------------------------
from src.models import predict_race_winner

# Example: Predict 2023 Australian Grand Prix
track = "Australian Grand Prix"
year = 2023
print(f"🏁 Predicted winner — {track} {year}")
print("="*60)
try:
    result = predict_race_winner(featured_df, track=track, year=year, model=xgb_model)
    print(result.to_string(index=False))
except Exception as e:
    print(f"Prediction error: {e}")
    print("(Ensure featured_df contains data for this race)")


In [ ]:
# Predict for another race
from src.models import predict_historical_race

track2 = "Monaco Grand Prix"
year2 = 2019
print(f"\n🏁 Historical prediction — {track2} {year2}")
print("="*60)
try:
    result2 = predict_historical_race(featured_df, track=track2, year=year2)
    print(result2.to_string(index=False))
except Exception as e:
    print(f"Prediction error: {e}")
